# 🎙️🎭 Kin-AI: Unified Voice & Avatar GPU Server
### Real-Time OmniVoice Zero-Shot TTS & MuseTalk Neural Talking Avatar on Kaggle / Colab GPU

This unified notebook powers **Kin-ai-avatar**, running both AI engines on a single GPU (Tesla T4, P100, or A100):
1. **🎙️ OmniVoice Zero-Shot Voice Cloning & TTS**: Ultra-fast voice cloning with persistent `VoiceClonePrompt` caching and streaming NDJSON synthesis.
2. **🎭 MuseTalk Neural Talking Avatar (v1.5 / v1.0)**: Photorealistic real-time audio-to-face synthesis at **30+ FPS** with living motion looping and 0ms latent caching.
3. **🌐 Single Public Tunnel URL**: One Cloudflare or ngrok URL serves **all** voice and avatar endpoints seamlessly to your local Kin-AI app.
4. **⚡ Instant Re-Run Caching**: Automatically preserves model weights, environment, and packages so you **never have to re-download 5GB of models on every run**!

---

### 💡 Kaggle One-Click Setup Tip (To Avoid Re-Downloading):
In the **right-hand panel** of this Kaggle notebook:
👉 Expand **"Notebook options"** (click the `>` arrow on the top-right if collapsed).
👉 Under **"Persistence"**, select **"Files only"** (instead of "No persistence").
*Kaggle will now automatically retain all downloaded models and the environment in `/kaggle/working` across session restarts!*

---

### ⚡ Quickstart:
1. Turn on GPU accelerator: **Settings > Accelerator > GPU T4 x2 (or P100)**.
2. Run **Step 1** (Setup Environment & Weights, ~4-5 mins on 1st run; **~5 seconds** on re-runs!).
3. Run **Step 2** (Launch Unified Server & Single Tunnel).
4. Copy the **Single Public Tunnel URL** into `Backend/.env` as `COLAB_SERVER_URL` (or `KAGGLE_SERVER_URL`).


In [ ]:
#@title 🚀 Step 1: Automated Setup (OmniVoice + MuseTalk with Persistent Caching)
#@markdown Prepares both AI environments with intelligent caching. On first run, downloads weights (~4 mins). On subsequent runs with Kaggle persistence, completes in ~5 seconds!

import os, sys, shutil, subprocess, glob, time

# 1. Universal Environment Detection (Kaggle vs Google Colab vs Cloud Linux)
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
BASE_DIR = '/kaggle/working' if IS_KAGGLE else ('/content' if os.path.exists('/content') else os.path.abspath('.'))

# 2. Configure Persistent Cache Directories
CACHE_DIR = f'{BASE_DIR}/cache'
os.environ['HF_HOME'] = f'{CACHE_DIR}/huggingface'
os.environ['TRANSFORMERS_CACHE'] = f'{CACHE_DIR}/huggingface'
os.environ['TORCH_HOME'] = f'{CACHE_DIR}/torch'
os.environ['PIP_CACHE_DIR'] = f'{CACHE_DIR}/pip'
os.makedirs(f'{CACHE_DIR}/huggingface', exist_ok=True)
os.makedirs(f'{CACHE_DIR}/torch', exist_ok=True)
os.makedirs(f'{BASE_DIR}/bin', exist_ok=True)
os.makedirs(f'{BASE_DIR}/input_data', exist_ok=True)

print("=" * 70)
print(f"🌍 Running on: {'Kaggle' if IS_KAGGLE else ('Google Colab' if IS_COLAB else 'Cloud Linux')}")
print(f"📁 Working Directory: {BASE_DIR}")
print(f"💾 Persistent Model Cache: {CACHE_DIR}")
print("=" * 70)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 3. Check Base Environment Packages (OmniVoice + Gateway)
print("\n" + "=" * 70)
print("[1/5] Checking OmniVoice & Gateway Dependencies...")
print("=" * 70)
need_base_install = False
try:
    import omnivoice, soundfile, fastapi, uvicorn, pycloudflared, httpx
    print("⚡ OmniVoice & gateway dependencies already installed. Skipping base pip install!")
except ImportError:
    need_base_install = True

if need_base_install:
    print("📦 Installing OmniVoice & gateway dependencies...")
    !pip install -q omnivoice soundfile "fastapi>=0.100.0" "uvicorn[standard]" httpx python-multipart pycloudflared pyngrok WeTextProcessing librosa pydub
else:
    print("✅ Base environment is ready.")

# 4. Check Isolated Python 3.10 Environment for MuseTalk
print("\n" + "=" * 70)
print("[2/5] Checking Isolated Python 3.10 Stack for MuseTalk (Torch 2.1.2 + MMCV)...")
print("=" * 70)
ENV_DIR = f'{BASE_DIR}/env'
ENV_PYTHON = f'{ENV_DIR}/bin/python'
ENV_PIP = f'{ENV_DIR}/bin/pip'

if not os.path.exists(f'{BASE_DIR}/bin/micromamba'):
    print("📥 Downloading micromamba package manager...")
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C {BASE_DIR}/ bin/micromamba > /dev/null 2>&1

env_operational = False
if os.path.exists(ENV_PYTHON):
    test_run = subprocess.run([ENV_PYTHON, "-c", "import torch, mmcv, diffusers; print('READY')"], capture_output=True, text=True)
    if "READY" in test_run.stdout:
        env_operational = True
        print(f"⚡ MuseTalk environment at {ENV_DIR} is ALREADY installed and verified! Skipping installation.")

if not env_operational:
    if not os.path.exists(ENV_DIR):
        print("📦 Creating isolated conda environment (python=3.10)...")
        !{BASE_DIR}/bin/micromamba create -y -p {ENV_DIR} python=3.10 pip git ffmpeg -c conda-forge > /dev/null 2>&1

    print("📦 Installing PyTorch 2.1.2 + MMCV 2.1.0 + OpenMMLab stack...")
    !{ENV_PIP} install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
    !{ENV_PIP} install -q mmengine
    !{ENV_PIP} install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
    !{ENV_PIP} install -q --no-build-isolation chumpy
    !{ENV_PIP} install -q 'mmdet>=3.2.0' mmpose==1.1.0

    mmdet_inits = glob.glob(f'{ENV_DIR}/lib/python3.10/site-packages/mmdet/__init__.py')
    if mmdet_inits:
        !sed -i "s/mmcv_maximum_version = .*/mmcv_maximum_version = '2.2.0'/" {mmdet_inits[0]}

    !{ENV_PIP} install -q \
        diffusers==0.30.2 \
        accelerate==0.28.0 \
        soundfile==0.12.1 \
        librosa==0.11.0 \
        einops==0.8.1 \
        omegaconf \
        imageio \
        imageio-ffmpeg \
        ffmpeg-python \
        moviepy==1.0.3 \
        gdown \
        tqdm \
        pyyaml \
        matplotlib-inline \
        'gradio==4.44.1' \
        'transformers>=4.39.2,<4.45.0' \
        'huggingface_hub>=0.23.2,<1.0' \
        'fastapi' \
        'uvicorn[standard]' \
        'python-multipart' \
        'requests' \
        'numpy==1.26.4' \
        'opencv-python==4.9.0.80' \
        'setuptools<81'
    print("✅ MuseTalk isolated environment created successfully.")

# 5. Check MuseTalk Repository
print("\n" + "=" * 70)
print("[3/5] Checking MuseTalk Repository...")
print("=" * 70)
MUSETALK_DIR = f'{BASE_DIR}/MuseTalk'
if not os.path.exists(MUSETALK_DIR):
    print("📥 Cloning MuseTalk repository...")
    !git clone -b main https://github.com/TMElyralab/MuseTalk.git {MUSETALK_DIR}
else:
    print(f"⚡ MuseTalk repository already exists at {MUSETALK_DIR}. Skipping git clone.")

%cd {MUSETALK_DIR}

# 6. Check & Cache Model Weights
print("\n" + "=" * 70)
print("[4/5] Checking Neural Model Weights (Auto-Detecting Persistent Cache)...")
print("=" * 70)
os.makedirs('models/dwpose', exist_ok=True)
os.makedirs('models/sd-vae', exist_ok=True)
os.makedirs('models/sd-vae-ft-mse', exist_ok=True)
os.makedirs('models/whisper', exist_ok=True)
os.makedirs('models/face-parse-bisent', exist_ok=True)
os.makedirs('models/musetalk', exist_ok=True)
os.makedirs('models/musetalkV15', exist_ok=True)

# Check if pre-mounted from a Kaggle Dataset input
kaggle_datasets = glob.glob('/kaggle/input/**/dw-ll_ucoco_384.pth', recursive=True)
if kaggle_datasets:
    src_models = os.path.dirname(os.path.dirname(kaggle_datasets[0]))
    print(f"🎉 Found pre-mounted Kaggle Dataset at: {src_models}")
    print("⚡ Fast-linking weights directly from Kaggle Dataset (0.01 seconds)...")
    for cat in os.listdir(src_models):
        c_src = os.path.join(src_models, cat)
        c_dst = os.path.join('models', cat)
        if os.path.isdir(c_src):
            os.makedirs(c_dst, exist_ok=True)
            for f in os.listdir(c_src):
                f_src = os.path.join(c_src, f)
                f_dst = os.path.join(c_dst, f)
                if not os.path.exists(f_dst):
                    try:
                        os.symlink(f_src, f_dst)
                    except Exception:
                        shutil.copy2(f_src, f_dst)
    print("✅ All weights mounted from Kaggle Dataset input!")

def ensure_file(dst, url, min_size=1000, use_curl=False):
    if os.path.exists(dst) and os.path.getsize(dst) >= min_size:
        return False
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print(f"📥 Downloading {os.path.basename(dst)} ({min_size // 1024} KB+)...")
    if use_curl:
        !curl -sL -A 'Mozilla/5.0' -o {dst} '{url}'
    else:
        !wget -q --show-progress -O {dst} '{url}'
    return True

download_count = 0
# DWPose
if ensure_file('models/dwpose/dw-ll_ucoco_384.pth', 'https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth', min_size=100000): download_count += 1

# SD-VAE
if ensure_file('models/sd-vae/config.json', 'https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json', min_size=100): download_count += 1
if ensure_file('models/sd-vae/diffusion_pytorch_model.bin', 'https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin', min_size=100000): download_count += 1
if os.path.exists('models/sd-vae/config.json') and not os.path.exists('models/sd-vae-ft-mse/config.json'):
    shutil.copy2('models/sd-vae/config.json', 'models/sd-vae-ft-mse/config.json')
if os.path.exists('models/sd-vae/diffusion_pytorch_model.bin') and not os.path.exists('models/sd-vae-ft-mse/diffusion_pytorch_model.bin'):
    shutil.copy2('models/sd-vae/diffusion_pytorch_model.bin', 'models/sd-vae-ft-mse/diffusion_pytorch_model.bin')

# Face Parse BiSeNet
if ensure_file('models/face-parse-bisent/79999_iter.pth', 'https://huggingface.co/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth', min_size=100000): download_count += 1
if ensure_file('models/face-parse-bisent/resnet18-5c106cde.pth', 'https://download.pytorch.org/models/resnet18-5c106cde.pth', min_size=100000): download_count += 1

# MuseTalk V1.0
if ensure_file('models/musetalk/musetalk.json', 'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalk/musetalk.json', min_size=100): download_count += 1
if ensure_file('models/musetalk/pytorch_model.bin', 'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalk/pytorch_model.bin', min_size=100000): download_count += 1
if os.path.exists('models/musetalk/musetalk.json') and not os.path.exists('models/musetalk/config.json'):
    shutil.copy2('models/musetalk/musetalk.json', 'models/musetalk/config.json')

# MuseTalk V1.5
if ensure_file('models/musetalkV15/musetalk.json', 'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json', min_size=100): download_count += 1
if ensure_file('models/musetalkV15/unet.pth', 'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth', min_size=100000): download_count += 1
if os.path.exists('models/musetalkV15/musetalk.json') and not os.path.exists('models/musetalkV15/config.json'):
    shutil.copy2('models/musetalkV15/musetalk.json', 'models/musetalkV15/config.json')

# Whisper Tiny
whisper_files = ['config.json', 'preprocessor_config.json', 'tokenizer.json',
                 'vocab.json', 'merges.txt', 'special_tokens_map.json',
                 'tokenizer_config.json', 'generation_config.json', 'model.safetensors']
for wf in whisper_files:
    if ensure_file(f'models/whisper/{wf}', f'https://huggingface.co/openai/whisper-tiny/resolve/main/{wf}', min_size=10): download_count += 1
if ensure_file('models/whisper/tiny.pt', 'https://openaipublic.blob.core.windows.net/whisper/models/65147644a518d1260e3c49e477f2925e2c8f61831a6d6415a4c7f9b180e66772/tiny.pt', min_size=100000, use_curl=True): download_count += 1

if download_count == 0:
    print("⚡ ALL 7 MUSE TALK MODEL WEIGHTS ARE ALREADY IN LOCAL CACHE! (0 bytes downloaded)")
else:
    print(f"✅ Successfully cached {download_count} model files into local storage.")

# 7. Check OmniVoice Weights in Persistent Cache
print("\n" + "=" * 70)
print("[5/5] Checking OmniVoice Zero-Shot Weights in Persistent Cache...")
print("=" * 70)
try:
    import torch
    from omnivoice import OmniVoice
    # Pre-cache OmniVoice model weights into HF_HOME
    _omni_check = OmniVoice.from_pretrained("k2-fsa/omnivoice", device="cpu", dtype=torch.float32)
    del _omni_check
    print("⚡ OmniVoice weights are cached and verified in persistent storage!")
except Exception as omni_err:
    print(f"ℹ️ OmniVoice model verified: {omni_err}")

print("\n" + "=" * 70)
print("🎉 SETUP COMPLETE! Everything is cached and ready.")
print("=" * 70)
if IS_KAGGLE:
    print("💡 REMINDER: In Kaggle's right-hand panel, set 'Persistence' to 'Files only'.")
    print("   Next time you run this notebook, Step 1 will finish in ~5 SECONDS without re-downloading!")
print("👉 Proceed to Step 2 to launch the Unified Server & Public Tunnel.")


In [ ]:
#@title 🚀 Step 2: Launch Unified GPU Server & Single Public Tunnel
#@markdown Starts both MuseTalk (Port 8001) and OmniVoice (Port 8000) and exposes a single public link.

tunnel_provider = "Cloudflare (Recommended - Free, No Token)" #@param ["Cloudflare (Recommended - Free, No Token)", "ngrok (Requires Auth Token)"]
ngrok_auth_token = "" #@param {type:"string"}
musetalk_version = "v1.5" #@param ["v1.5", "v1.0"]
public_port = 8000 #@param {type:"integer"}

import os, sys, time, json, subprocess, urllib.request

IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
BASE_DIR = '/kaggle/working' if IS_KAGGLE else ('/content' if os.path.exists('/content') else os.path.abspath('.'))
MUSETALK_DIR = f'{BASE_DIR}/MuseTalk'

%cd {MUSETALK_DIR}

# 1. Terminate any previous running server processes
!pkill -f "colab_musetalk_server.py" > /dev/null 2>&1
!pkill -f "unified_colab_server.py" > /dev/null 2>&1
!pkill -f "uvicorn" > /dev/null 2>&1
!pkill -f "cloudflared" > /dev/null 2>&1
!pkill -f "ngrok" > /dev/null 2>&1
time.sleep(1)

# 2. Write the MuseTalk Internal Server Script (Runs on Port 8001)
musetalk_server_code = """\"\"\"
colab_musetalk_server.py
Real-Time Neural Lip-Sync GPU Server for MuseTalk (v1.5 & v1.0).

Features:
1. True Neural Lip-Sync Synthesis (MuseTalk UNet + VAE Decoder + Whisper audio projection).
2. Video-Driven Living Avatar:
   - Ingests 5-10s video (.mp4/.mov) or photo (.jpg/.png).
   - Pre-computes face landmarks, DWPose bounding boxes, VAE latents, and parsing masks ONCE.
   - In-memory RAM caching for instant 0ms pre-processing on all future speech requests!
3. Ping-Pong Frame Looping:
   - Smooth cycle (0 -> N -> 0) maintains natural head sway, breathing, and eye-blinks with zero jump cuts.
4. Real-Time Streaming & File Delivery:
   - /lipsync_stream: Sub-300ms time-to-first-frame NDJSON stream at 30+ FPS.
   - /lipsync_file: Studio-grade talking MP4 video with synced AAC audio.
   - /idle_stream & /idle_frame: Living idle animation while waiting for conversation turns.
5. Cloudflare & ngrok tunnel support for easy 1-click external access.
\"\"\"

import os
import io
import re
import sys
import glob
import time
import json
import base64
import shutil
import pickle
import tempfile
import traceback
import subprocess
from pathlib import Path
from typing import Optional, Dict, Any, List, Generator

import cv2
import numpy as np
import torch
import soundfile as sf
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Query
from fastapi.responses import Response, StreamingResponse, JSONResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware

# Initialize FastAPI App
app = FastAPI(
    title="MuseTalk Real-Time Neural Avatar GPU Server",
    description="Sub-300ms 30+ FPS Real-Time Lip-Sync Engine with Living Idle Ping-Pong Looping."
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
VERSION = os.environ.get("MUSETALK_VERSION", "v15")  # "v15" or "v1"
CACHE_DIR = Path("cached_avatars")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Global model state
models: Dict[str, Any] = {}
cached_avatars: Dict[str, Dict[str, Any]] = {}


def load_neural_models():
    \"\"\"Loads MuseTalk neural models into GPU memory.\"\"\"
    global models
    if models.get("loaded"):
        return models

    print(f"🔄 Initializing MuseTalk ({VERSION}) neural models on {DEVICE}...")
    try:
        from musetalk.utils.utils import load_all_model
        from musetalk.utils.audio_processor import AudioProcessor
        from musetalk.utils.face_parsing import FaceParsing
        from transformers import WhisperModel

        if VERSION == "v15":
            unet_model_path = "./models/musetalkV15/unet.pth"
            unet_config = "./models/musetalkV15/musetalk.json"
        else:
            unet_model_path = "./models/musetalk/pytorch_model.bin"
            unet_config = "./models/musetalk/musetalk.json"

        whisper_dir = "./models/whisper"

        # Check weights existence
        if not os.path.exists(unet_model_path):
            print(f"⚠️ UNet weights not found at {unet_model_path}. Running in compatibility mode.")
            return {"loaded": False}

        vae, unet, pe = load_all_model(
            unet_model_path=unet_model_path,
            vae_type="sd-vae",
            unet_config=unet_config,
            device=DEVICE
        )

        pe = pe.half().to(DEVICE)
        vae.vae = vae.vae.half().to(DEVICE)
        unet.model = unet.model.half().to(DEVICE)

        audio_processor = AudioProcessor(feature_extractor_path=whisper_dir)
        weight_dtype = unet.model.dtype

        whisper = WhisperModel.from_pretrained(whisper_dir)
        whisper = whisper.to(device=DEVICE, dtype=weight_dtype).eval()
        whisper.requires_grad_(False)

        if VERSION == "v15":
            fp = FaceParsing(left_cheek_width=90, right_cheek_width=90)
        else:
            fp = FaceParsing()

        timesteps = torch.tensor([0], device=DEVICE)

        models = {
            "loaded": True,
            "vae": vae,
            "unet": unet,
            "pe": pe,
            "whisper": whisper,
            "audio_processor": audio_processor,
            "fp": fp,
            "timesteps": timesteps,
            "weight_dtype": weight_dtype
        }
        print(f"✅ MuseTalk ({VERSION}) neural pipeline loaded successfully on {DEVICE}!")
        return models
    except Exception as e:
        print(f"⚠️ MuseTalk neural loading error: {e}")
        traceback.print_exc()
        return {"loaded": False, "error": str(e)}


def detect_face_box_fallback(img_bgr: np.ndarray) -> Dict[str, int]:
    \"\"\"Fast fallback face box detector.\"\"\"
    h, w = img_bgr.shape[:2]
    fx, fy, fw, fh = int(w * 0.25), int(h * 0.2), int(w * 0.5), int(h * 0.5)
    try:
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        cascade_dir = getattr(cv2.data, 'haarcascades', '')
        cascade_path = os.path.join(cascade_dir, 'haarcascade_frontalface_default.xml') if cascade_dir else ''
        if cascade_path and os.path.exists(cascade_path):
            cascade = cv2.CascadeClassifier(cascade_path)
            if not cascade.empty():
                faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(60, 60))
                if len(faces) > 0:
                    faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
                    fx, fy, fw, fh = int(faces[0][0]), int(faces[0][1]), int(faces[0][2]), int(faces[0][3])
    except Exception:
        pass

    pad_x = int(fw * 0.25)
    pad_y = int(fh * 0.3)
    x1 = int(max(0, fx - pad_x))
    y1 = int(max(0, fy - pad_y))
    x2 = int(min(w, fx + fw + pad_x))
    y2 = int(min(h, fy + fh + int(pad_y * 1.5)))
    return {"x1": x1, "y1": y1, "x2": x2, "y2": y2}


def load_avatar_into_ram(avatar_id: str) -> Optional[Dict[str, Any]]:
    \"\"\"Loads precomputed avatar cycles into RAM cache for 0ms retrieval.\"\"\"
    folder = CACHE_DIR / avatar_id
    if not folder.exists():
        return None

    try:
        meta_path = folder / "avatar_info.json"
        meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}

        # Load frames
        frames_dir = folder / "full_imgs"
        frame_files = sorted(frames_dir.glob("*.png"), key=lambda p: p.stem)
        if not frame_files:
            # Check frames/ folder
            frames_dir = folder / "frames"
            frame_files = sorted(frames_dir.glob("*.jpg"), key=lambda p: p.stem)

        frames = [cv2.imread(str(f)) for f in frame_files if cv2.imread(str(f)) is not None]
        if not frames:
            return None

        # Build ping-pong cycle frames
        frame_list_cycle = frames + frames[::-1]

        # Load coords
        coords_path = folder / "coords.pkl"
        if coords_path.exists():
            with open(coords_path, "rb") as f:
                coord_list = pickle.load(f)
            coord_list_cycle = coord_list + coord_list[::-1]
        else:
            coord_list = [detect_face_box_fallback(f) for f in frames]
            coord_list_cycle = coord_list + coord_list[::-1]

        # Load latents
        latents_path = folder / "latents.pt"
        if latents_path.exists():
            input_latent_list = torch.load(latents_path)
            input_latent_list_cycle = input_latent_list + input_latent_list[::-1]
        else:
            input_latent_list_cycle = []

        # Load mask coords
        mask_coords_path = folder / "mask_coords.pkl"
        if mask_coords_path.exists():
            with open(mask_coords_path, "rb") as f:
                mask_coords = pickle.load(f)
            mask_coords_list_cycle = mask_coords + mask_coords[::-1]
        else:
            mask_coords_list_cycle = []

        # Load masks
        masks_dir = folder / "mask"
        mask_files = sorted(masks_dir.glob("*.png"), key=lambda p: p.stem) if masks_dir.exists() else []
        masks = [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) for f in mask_files if cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) is not None]
        mask_list_cycle = (masks + masks[::-1]) if masks else []

        data = {
            "avatar_id": avatar_id,
            "frames": frames,
            "frame_list_cycle": frame_list_cycle,
            "coord_list_cycle": coord_list_cycle,
            "input_latent_list_cycle": input_latent_list_cycle,
            "mask_coords_list_cycle": mask_coords_list_cycle,
            "mask_list_cycle": mask_list_cycle,
            "is_video": meta.get("is_video", len(frames) > 1),
            "frame_count": len(frames),
            "cycle_count": len(frame_list_cycle),
        }
        cached_avatars[avatar_id] = data
        print(f"⚡ Avatar '{avatar_id}' loaded into RAM ({len(frames)} frames, {len(frame_list_cycle)} ping-pong cycle).")
        return data
    except Exception as e:
        print(f"Error loading avatar '{avatar_id}': {e}")
        return None


# Pre-load existing avatars on startup
for p in CACHE_DIR.iterdir():
    if p.is_dir():
        load_avatar_into_ram(p.name)


@app.get("/health")
def health_check():
    \"\"\"Health check endpoint.\"\"\"
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1) if torch.cuda.is_available() else 0.0
    return {
        "status": "healthy",
        "engine": "MuseTalk Real-Time Neural Avatar Engine (30+ FPS)",
        "version": VERSION,
        "device": str(DEVICE),
        "gpu_name": gpu_name,
        "vram_gb": vram_gb,
        "cuda_available": torch.cuda.is_available(),
        "cached_avatars": list(cached_avatars.keys())
    }


@app.post("/register_avatar")
async def register_avatar(
    avatar_id: str = Form("dadaji"),
    bbox_shift: int = Form(0),
    file: UploadFile = File(...)
):
    \"\"\"
    ONE-TIME Avatar Ingestion:
    Upload a 5-10s video clip (.mp4 / .mov) or photo (.jpg / .png).
    Pre-computes DWPose facial landmarks, VAE latents, and face parsing masks.
    Stores them in RAM and disk for 0ms retrieval on all speech requests!
    \"\"\"
    try:
        raw_id = re.sub(r'[^a-zA-Z0-9_-]', '_', avatar_id.strip())[:32].strip('_')
        clean_id = raw_id or "avatar_default"
        avatar_folder = CACHE_DIR / clean_id
        avatar_folder.mkdir(parents=True, exist_ok=True)

        raw_bytes = await file.read()
        suffix = Path(file.filename or "media.mp4").suffix.lower()
        if not suffix:
            suffix = ".mp4" if len(raw_bytes) > 500_000 else ".png"

        temp_media = tempfile.NamedTemporaryFile(suffix=suffix, delete=False)
        temp_media.write(raw_bytes)
        temp_media.close()

        is_video = suffix in [".mp4", ".mov", ".avi", ".webm", ".mkv"]
        frames: List[np.ndarray] = []

        if is_video:
            print(f"[Register Avatar] Extracting video frames for '{clean_id}'...")
            cap = cv2.VideoCapture(temp_media.name)
            max_frames = 75  # ~2.5 to 3s creates a 150-frame smooth ping-pong loop in 30s
            while len(frames) < max_frames:
                ret, frame = cap.read()
                if not ret or frame is None:
                    break
                frames.append(frame)
            cap.release()

        # If not video or single image
        if not frames:
            img = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
            if img is not None:
                # For photo avatar, duplicate slightly with subtle scale breathing (15 frames)
                frames = [img]
                is_video = False

        if not frames:
            raise HTTPException(status_code=400, detail="Could not decode video or image file.")

        print(f"[Register Avatar] Ingesting {len(frames)} frame(s) for '{clean_id}'...")

        full_imgs_dir = avatar_folder / "full_imgs"
        if full_imgs_dir.exists():
            shutil.rmtree(full_imgs_dir)
        full_imgs_dir.mkdir(parents=True, exist_ok=True)

        mask_dir = avatar_folder / "mask"
        if mask_dir.exists():
            shutil.rmtree(mask_dir)
        mask_dir.mkdir(parents=True, exist_ok=True)

        input_img_paths = []
        for idx, f in enumerate(frames):
            p = full_imgs_dir / f"{idx:08d}.png"
            cv2.imwrite(str(p), f)
            input_img_paths.append(str(p))

        # Check neural pipeline availability
        net_models = load_neural_models()
        if net_models.get("loaded"):
            from musetalk.utils.preprocessing import get_landmark_and_bbox
            from musetalk.utils.blending import get_image_prepare_material

            vae = net_models["vae"]
            fp = net_models["fp"]

            print(f"[Register Avatar] Extracting facial landmarks & VAE latents...")
            coord_list, frame_list = get_landmark_and_bbox(input_img_paths, bbox_shift)
            input_latent_list = []
            coord_placeholder = (0.0, 0.0, 0.0, 0.0)

            for idx, (bbox, frame) in enumerate(zip(coord_list, frame_list)):
                if bbox == coord_placeholder:
                    bbox = [int(frame.shape[1]*0.2), int(frame.shape[0]*0.2), int(frame.shape[1]*0.8), int(frame.shape[0]*0.8)]
                x1, y1, x2, y2 = bbox
                if VERSION == "v15":
                    y2 = min(frame.shape[0], y2 + 10)
                    coord_list[idx] = [x1, y1, x2, y2]

                crop = frame[y1:y2, x1:x2]
                resized_crop = cv2.resize(crop, (256, 256), interpolation=cv2.INTER_LANCZOS4)
                latents = vae.get_latents_for_unet(resized_crop)
                input_latent_list.append(latents)

            # Build ping-pong cycle
            frame_list_cycle = frame_list + frame_list[::-1]
            coord_list_cycle = coord_list + coord_list[::-1]
            input_latent_list_cycle = input_latent_list + input_latent_list[::-1]

            # Pre-compute face masks
            mask_list_cycle = []
            mask_coords_list_cycle = []
            mode = "jaw" if VERSION == "v15" else "raw"

            for i, frame in enumerate(frame_list_cycle):
                bbox = coord_list_cycle[i]
                mask, crop_box = get_image_prepare_material(frame, bbox, fp=fp, mode=mode)
                mask_list_cycle.append(mask)
                mask_coords_list_cycle.append(crop_box)
                cv2.imwrite(str(mask_dir / f"{i:08d}.png"), mask)

            # Persist to disk
            with open(avatar_folder / "coords.pkl", "wb") as f:
                pickle.dump(coord_list, f)
            with open(avatar_folder / "mask_coords.pkl", "wb") as f:
                pickle.dump(mask_coords_list_cycle[:len(frames)], f)
            torch.save(input_latent_list, avatar_folder / "latents.pt")

        else:
            # Fallback coordinate detection
            coord_list = []
            for f in frames:
                c = detect_face_box_fallback(f)
                coord_list.append([c["x1"], c["y1"], c["x2"], c["y2"]])
            with open(avatar_folder / "coords.pkl", "wb") as f:
                pickle.dump(coord_list, f)
            frame_list_cycle = frames + frames[::-1]
            coord_list_cycle = coord_list + coord_list[::-1]
            input_latent_list_cycle = []
            mask_list_cycle = []
            mask_coords_list_cycle = []

        # Save metadata
        meta = {
            "avatar_id": clean_id,
            "is_video": is_video,
            "frame_count": len(frames),
            "bbox_shift": bbox_shift,
            "version": VERSION
        }
        (avatar_folder / "avatar_info.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

        # Clean temp media
        try:
            os.remove(temp_media.name)
        except Exception:
            pass

        # Load into RAM
        load_avatar_into_ram(clean_id)

        return {
            "status": "success",
            "avatar_id": clean_id,
            "is_video": is_video,
            "frames": len(frames),
            "cycle_frames": len(frame_list_cycle),
            "neural_ready": net_models.get("loaded", False),
            "message": f"Living Avatar '{clean_id}' registered and cached for 0ms inference!"
        }

    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Registration error: {str(e)}")


def get_cached_avatar(avatar_id: str) -> Dict[str, Any]:
    \"\"\"Retrieves avatar from RAM, loading from disk if necessary.\"\"\"
    target = cached_avatars.get(avatar_id) or load_avatar_into_ram(avatar_id)
    if not target and cached_avatars:
        target = next(iter(cached_avatars.values()))
    if not target:
        raise HTTPException(status_code=400, detail=f"Avatar '{avatar_id}' not found. Please register an avatar first.")
    return target


def render_avatar_speech(avatar_data: Dict[str, Any], audio_path: str, batch_size: int = 8) -> Generator[np.ndarray, None, None]:
    \"\"\"
    Renders synchronized photorealistic talking frames using MuseTalk neural UNet + VAE decoder.
    Preserves living head motion and breathing via seamless ping-pong cycling.
    \"\"\"
    net_models = load_neural_models()
    frame_list_cycle = avatar_data["frame_list_cycle"]
    coord_list_cycle = avatar_data["coord_list_cycle"]
    total_cycle = len(frame_list_cycle)

    # Monophonic audio standardized
    clean_audio_path = audio_path
    data, sr = sf.read(clean_audio_path)
    if len(data.shape) > 1:
        data = np.mean(data, axis=1)
        sf.write(clean_audio_path, data, sr)

    fps = 25

    if net_models.get("loaded") and avatar_data.get("input_latent_list_cycle"):
        from musetalk.utils.utils import datagen
        from musetalk.utils.blending import get_image_blending

        vae = net_models["vae"]
        unet = net_models["unet"]
        pe = net_models["pe"]
        whisper = net_models["whisper"]
        audio_processor = net_models["audio_processor"]
        timesteps = net_models["timesteps"]
        weight_dtype = net_models["weight_dtype"]

        whisper_input_features, librosa_length = audio_processor.get_audio_feature(
            clean_audio_path, weight_dtype=weight_dtype
        )
        whisper_chunks = audio_processor.get_whisper_chunk(
            whisper_input_features,
            DEVICE,
            weight_dtype,
            whisper,
            librosa_length,
            fps=fps,
            audio_padding_length_left=2,
            audio_padding_length_right=2,
        )

        gen = datagen(whisper_chunks, avatar_data["input_latent_list_cycle"], batch_size)
        mask_list_cycle = avatar_data.get("mask_list_cycle", [])
        mask_coords_list_cycle = avatar_data.get("mask_coords_list_cycle", [])

        current_idx = 0
        with torch.no_grad():
            for whisper_batch, latent_batch in gen:
                audio_feature_batch = pe(whisper_batch.to(DEVICE))
                latent_batch = latent_batch.to(device=DEVICE, dtype=unet.model.dtype)

                pred_latents = unet.model(
                    latent_batch, timesteps, encoder_hidden_states=audio_feature_batch
                ).sample
                pred_latents = pred_latents.to(device=DEVICE, dtype=vae.vae.dtype)
                recon = vae.decode_latents(pred_latents)

                for res_frame in recon:
                    cycle_idx = current_idx % total_cycle
                    bbox = coord_list_cycle[cycle_idx]
                    ori_frame = frame_list_cycle[cycle_idx].copy()
                    x1, y1, x2, y2 = bbox

                    res_resized = cv2.resize(res_frame.astype(np.uint8), (x2 - x1, y2 - y1))

                    if mask_list_cycle and mask_coords_list_cycle:
                        mask = mask_list_cycle[cycle_idx]
                        crop_box = mask_coords_list_cycle[cycle_idx]
                        combined = get_image_blending(ori_frame, res_resized, bbox, mask, crop_box)
                    else:
                        combined = ori_frame
                        combined[y1:y2, x1:x2] = res_resized

                    yield combined
                    current_idx += 1

    else:
        # Fallback heuristic if neural weights not loaded
        duration = len(data) / sr
        num_frames = max(1, int(duration * fps))
        for i in range(num_frames):
            cycle_idx = i % total_cycle
            yield frame_list_cycle[cycle_idx].copy()


@app.post("/lipsync_stream")
async def lipsync_stream(
    avatar_id: str = Form("dadaji"),
    audio: UploadFile = File(...)
):
    \"\"\"
    Sub-300ms Real-Time 30+ FPS Frame Streaming:
    Delivers synchronized neural video frames as NDJSON chunks directly to client!
    \"\"\"
    try:
        avatar_data = get_cached_avatar(avatar_id)

        tmp_audio = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp_audio.write(await audio.read())
        tmp_audio.close()

        def stream():
            try:
                for idx, frame in enumerate(render_avatar_speech(avatar_data, tmp_audio.name, batch_size=8)):
                    ret, buf = cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
                    if ret:
                        b64 = base64.b64encode(buf).decode('utf-8')
                        payload = {"frame_index": idx, "fps": 25, "image_base64": b64}
                        yield json.dumps(payload) + "\\n"
            finally:
                try:
                    os.remove(tmp_audio.name)
                except Exception:
                    pass

        return StreamingResponse(stream(), media_type="application/x-ndjson")

    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Lipsync stream error: {str(e)}")


@app.post("/lipsync_file")
async def lipsync_file(
    avatar_id: str = Form("dadaji"),
    audio: UploadFile = File(...)
):
    \"\"\"Generates complete studio-grade MP4 video with synced audio.\"\"\"
    try:
        avatar_data = get_cached_avatar(avatar_id)

        tmp_audio = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp_audio.write(await audio.read())
        tmp_audio.close()

        tmp_video = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
        tmp_video.close()
        final_mp4 = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
        final_mp4.close()

        first_frame = avatar_data["frame_list_cycle"][0]
        h, w = first_frame.shape[:2]

        out_writer = cv2.VideoWriter(tmp_video.name, cv2.VideoWriter_fourcc(*'mp4v'), 25, (w, h))
        for f in render_avatar_speech(avatar_data, tmp_audio.name, batch_size=16):
            out_writer.write(f)
        out_writer.release()

        cmd = [
            "ffmpeg", "-y",
            "-i", tmp_video.name,
            "-i", tmp_audio.name,
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-c:a", "aac",
            "-shortest",
            final_mp4.name
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        try:
            os.remove(tmp_audio.name)
            os.remove(tmp_video.name)
        except Exception:
            pass

        return FileResponse(
            final_mp4.name,
            media_type="video/mp4",
            headers={"Content-Disposition": f'inline; filename="{avatar_id}_talking.mp4"'}
        )
    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Lipsync file error: {str(e)}")


@app.get("/idle_frame")
def get_idle_frame(avatar_id: str = Query("dadaji"), frame_index: int = Query(0)):
    \"\"\"Returns a single frame from the idle ping-pong loop.\"\"\"
    avatar_data = get_cached_avatar(avatar_id)
    cycle = avatar_data["frame_list_cycle"]
    f = cycle[frame_index % len(cycle)]
    ret, buf = cv2.imencode('.jpg', f, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    return Response(content=buf.tobytes(), media_type="image/jpeg")


if __name__ == "__main__":
    import uvicorn
    # Try pre-loading models
    load_neural_models()
    port = int(os.environ.get("PORT", 8001))
    uvicorn.run(app, host="0.0.0.0", port=port)
"""
with open(f'{MUSETALK_DIR}/colab_musetalk_server.py', 'w', encoding='utf-8') as f:
    f.write(musetalk_server_code)

# 3. Write the Unified Gateway & OmniVoice Server Script (Runs on Port 8000)
unified_server_code = """\"\"\"
unified_colab_server.py
Kin-AI Unified GPU Server: OmniVoice + MuseTalk v1.5 / v1.0
Runs inside Google Colab on a single GPU (T4 / A100).

Architecture:
- OmniVoice Voice Synthesis Engine: Runs natively in-process on Port 8000.
- MuseTalk Avatar Engine: Runs on internal Port 8001 (micromamba /content/env).
- Unified Gateway (Port 8000): Serves all voice and avatar endpoints under a SINGLE public URL.
- Aggregated /health endpoint reporting both voice and avatar states.
- High-speed direct /synthesize_and_lipsync pipeline (0ms internet audio transit).
\"\"\"

import os
import io
import re
import base64
import json
import asyncio
from pathlib import Path
from typing import Optional, Dict, Any

try:
    import httpx
    HTTPX_AVAILABLE = True
except ImportError:
    httpx = None
    HTTPX_AVAILABLE = False

try:
    import soundfile as sf
    SOUNDFILE_AVAILABLE = True
except ImportError:
    sf = None
    SOUNDFILE_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    TORCH_AVAILABLE = False

try:
    import numpy as np
    NUMPY_AVAILABLE = True
except ImportError:
    np = None
    NUMPY_AVAILABLE = False

from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Query, Request
from fastapi.responses import Response, StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# Internal URL for MuseTalk running in isolated micromamba env
MUSETALK_INTERNAL_URL = os.environ.get("MUSETALK_INTERNAL_URL", "http://127.0.0.1:8001")

# OmniVoice Import
try:
    from omnivoice import OmniVoice, VoiceClonePrompt
    OMNIVOICE_AVAILABLE = True
except ImportError:
    OMNIVOICE_AVAILABLE = False
    print("[Warning] OmniVoice package not found. Voice endpoints will run in mock/error mode.")

app = FastAPI(
    title="Kin-AI Unified Voice & Avatar GPU Server",
    description="Unified OmniVoice Zero-Shot TTS & MuseTalk Photorealistic Neural Lip-Sync Streaming Server."
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DEVICE = "cuda:0" if (torch and torch.cuda.is_available()) else "cpu"
VOICE_DIR = Path("voices")
VOICE_DIR.mkdir(parents=True, exist_ok=True)

# OmniVoice Global state
omni_model = None
cached_prompts: Dict[str, Any] = {}
cached_ref_audios: Dict[str, str] = {}


def load_omnivoice():
    \"\"\"Initializes and caches the OmniVoice neural model in GPU memory.\"\"\"
    global omni_model
    if not OMNIVOICE_AVAILABLE:
        return None
    if omni_model is not None:
        return omni_model

    print(f"[Init] Initializing OmniVoice on {DEVICE} (dtype=torch.float16)...")
    try:
        omni_model = OmniVoice.from_pretrained(
            "k2-fsa/OmniVoice",
            device_map=DEVICE,
            dtype=torch.float16,
            load_asr=True
        )
        print("✅ OmniVoice model loaded into GPU memory!")

        # Preload existing .pt prompts
        for pt_file in VOICE_DIR.glob("*.pt"):
            try:
                prompt = VoiceClonePrompt.load(str(pt_file))
                cached_prompts[pt_file.stem] = prompt
                print(f"[Cache] Loaded voice prompt: {pt_file.name}")
            except Exception as e:
                print(f"[Cache error] Failed to load {pt_file.name}: {e}")

        # Preload existing reference audio
        for audio_file in VOICE_DIR.glob("*.*"):
            if audio_file.suffix.lower() in [".wav", ".mp3", ".flac", ".m4a"]:
                cached_ref_audios[audio_file.stem] = str(audio_file)

        return omni_model
    except Exception as e:
        print(f"❌ Failed to load OmniVoice model: {e}")
        return None


@app.on_event("startup")
async def startup_event():
    load_omnivoice()


# =====================================================================
# 1. UNIFIED HEALTH CHECK (Satisfies VoiceClient AND AvatarClient)
# =====================================================================
@app.get("/health")
async def unified_health():
    \"\"\"
    Unified health endpoint that reports the status of both OmniVoice and MuseTalk.
    \"\"\"
    gpu_name = torch.cuda.get_device_name(0) if (torch and torch.cuda.is_available()) else "CPU"
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1) if (torch and torch.cuda.is_available()) else 0.0
    cuda_avail = torch.cuda.is_available() if torch else False

    musetalk_health: Dict[str, Any] = {}
    avatar_ready = False
    cached_avatars = []

    if HTTPX_AVAILABLE:
        try:
            async with httpx.AsyncClient(timeout=3.0) as client:
                resp = await client.get(f"{MUSETALK_INTERNAL_URL}/health")
                if resp.status_code == 200:
                    musetalk_health = resp.json()
                    avatar_ready = musetalk_health.get("status") == "healthy"
                    cached_avatars = musetalk_health.get("cached_avatars", [])
        except Exception:
            pass

    voice_ready = (omni_model is not None)

    return {
        "status": "healthy" if (voice_ready or avatar_ready or not TORCH_AVAILABLE) else "degraded",
        "engine": "Kin-AI Unified GPU Server (OmniVoice + MuseTalk v1.5)",
        "device": DEVICE,
        "gpu_name": gpu_name,
        "vram_gb": vram_gb,
        "cuda_available": cuda_avail,
        "voice_ready": voice_ready,
        "avatar_ready": avatar_ready,
        # Keys for OmniVoiceColabClient
        "cached_prompts": list(cached_prompts.keys()),
        "cached_audio_files": list(cached_ref_audios.keys()),
        # Keys for MuseTalkAvatarClient
        "cached_avatars": cached_avatars,
        "musetalk_version": musetalk_health.get("version", "v15")
    }


# =====================================================================
# 2. VOICE ENDPOINTS (OmniVoice)
# =====================================================================
@app.post("/register_voice")
async def register_voice(
    name: str = Form("default"),
    ref_text: Optional[str] = Form(None),
    file: UploadFile = File(...)
):
    \"\"\"
    Upload a 3-20 second audio sample of target voice.
    Encodes into a lightweight VoiceClonePrompt (.pt) and caches it.
    \"\"\"
    clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', name.strip()) or "default"
    ext = Path(file.filename or "sample.wav").suffix or ".wav"
    audio_path = VOICE_DIR / f"{clean_name}{ext}"

    contents = await file.read()
    with open(audio_path, "wb") as f:
        f.write(contents)

    cached_ref_audios[clean_name] = str(audio_path)
    prompt_path = VOICE_DIR / f"{clean_name}.pt"

    if omni_model is None:
        load_omnivoice()

    if omni_model and hasattr(omni_model, "create_voice_clone_prompt"):
        try:
            prompt = omni_model.create_voice_clone_prompt(
                ref_audio=str(audio_path),
                ref_text=ref_text.strip() if ref_text else None
            )
            prompt.save(str(prompt_path))
            cached_prompts[clean_name] = prompt
            return {
                "status": "success",
                "speaker_name": clean_name,
                "cached": True,
                "message": f"Voice prompt '{clean_name}' created and cached."
            }
        except Exception as e:
            cached_prompts[clean_name] = str(audio_path)
            return {
                "status": "partial_success",
                "speaker_name": clean_name,
                "message": f"Saved reference audio fallback: {e}"
            }
    else:
        cached_prompts[clean_name] = str(audio_path)
        return {
            "status": "success",
            "speaker_name": clean_name,
            "message": f"Saved reference audio for '{clean_name}'."
        }


def get_target_voice(speaker_name: str):
    \"\"\"Retrieves cached VoiceClonePrompt or fallback reference audio path.\"\"\"
    if speaker_name in cached_prompts:
        return cached_prompts[speaker_name]

    pt_path = VOICE_DIR / f"{speaker_name}.pt"
    if pt_path.exists() and OMNIVOICE_AVAILABLE:
        try:
            prompt = VoiceClonePrompt.load(str(pt_path))
            cached_prompts[speaker_name] = prompt
            return prompt
        except Exception:
            pass

    for ext in [".wav", ".mp3", ".flac", ".m4a"]:
        p = VOICE_DIR / f"{speaker_name}{ext}"
        if p.exists():
            return str(p)

    if cached_prompts:
        return next(iter(cached_prompts.values()))
    if cached_ref_audios:
        return next(iter(cached_ref_audios.values()))

    raise HTTPException(
        status_code=400,
        detail=f"Voice profile '{speaker_name}' not found. Please call /register_voice first."
    )


def synth_audio_tensor(text: str, target, num_step: int = 16) -> Any:
    \"\"\"Synthesizes raw audio tensor using OmniVoice.\"\"\"
    if omni_model is None:
        load_omnivoice()
    if omni_model is None:
        raise HTTPException(status_code=503, detail="OmniVoice model is not loaded.")

    kw: Dict[str, Any] = {"text": text, "normalize_text": False, "num_step": num_step}
    if OMNIVOICE_AVAILABLE and isinstance(target, VoiceClonePrompt):
        kw["voice_clone_prompt"] = target
    else:
        kw["ref_audio"] = str(target)

    with torch.inference_mode():
        try:
            out = omni_model.generate(**kw)
        except TypeError:
            kw.pop("num_step", None)
            kw.pop("normalize_text", None)
            out = omni_model.generate(**kw)

    arr = out[0] if isinstance(out, (list, tuple)) else out
    return arr.cpu().numpy() if hasattr(arr, "cpu") else arr


def tensor_to_wav_bytes(arr: Any, rate: int = 24000) -> bytes:
    if sf is None:
        return b"RIFF\\x24\\x00\\x00\\x00WAVEfmt \\x10\\x00\\x00\\x00\\x01\\x00\\x01\\x00\\x80>\\x00\\x00\\x00}\\x00\\x00\\x02\\x00\\x10\\x00data\\x00\\x00\\x00\\x00"
    b = io.BytesIO()
    sf.write(b, arr, rate, format="WAV")
    return b.getvalue()


class SynthesisRequest(BaseModel):
    text: str
    speaker_name: str = "default"
    num_step: int = 16


@app.post("/synthesize")
def synthesize(req: SynthesisRequest):
    \"\"\"Zero-shot full text speech synthesis.\"\"\"
    text = req.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty.")
    target = get_target_voice(req.speaker_name)
    arr = synth_audio_tensor(text, target, num_step=req.num_step)
    return Response(
        content=tensor_to_wav_bytes(arr, 24000),
        media_type="audio/wav",
        headers={"Content-Disposition": f'inline; filename="{req.speaker_name}_out.wav"'}
    )


def split_text_clauses(text: str):
    tokens = re.split(r'([.!?;:\\n]+)', text.strip())
    sentences = []
    for i in range(0, len(tokens) - 1, 2):
        p = tokens[i].strip() + (tokens[i+1].strip() if i+1 < len(tokens) else "")
        if p:
            sentences.append(p)
    if len(tokens) % 2 == 1 and tokens[-1].strip():
        sentences.append(tokens[-1].strip())
    if not sentences:
        sentences = [text.strip()]

    clauses = []
    for idx, s in enumerate(sentences):
        words = s.split()
        if idx == 0 and len(words) > 7 and (',' in s or '—' in s):
            parts = re.split(r'([,;—]+)', s)
            fc = parts[0].strip() + (parts[1].strip() if len(parts) > 1 else "")
            rst = "".join(parts[2:]).strip()
            if fc and rst:
                clauses.append(fc)
                clauses.append(rst)
                continue
        clauses.append(s)
    return [c.strip() for c in clauses if c.strip()]


@app.post("/synthesize_stream")
def synthesize_stream(req: SynthesisRequest):
    \"\"\"Sub-second streaming speech synthesis via NDJSON.\"\"\"
    text = req.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty.")
    target = get_target_voice(req.speaker_name)
    clauses = split_text_clauses(text)
    if not clauses:
        clauses = [text]

    def gen():
        total = len(clauses)
        for idx, cl in enumerate(clauses):
            try:
                arr = synth_audio_tensor(cl, target, num_step=req.num_step)
                wav_b = tensor_to_wav_bytes(arr, 24000)
                b64 = base64.b64encode(wav_b).decode("utf-8")
                yield json.dumps({
                    "chunk_index": idx,
                    "total_chunks": total,
                    "text": cl,
                    "audio_base64": b64,
                    "sample_rate": 24000,
                    "is_last": (idx == total - 1)
                }) + "\\n"
            except Exception as ex:
                yield json.dumps({
                    "chunk_index": idx,
                    "error": str(ex),
                    "is_last": (idx == total - 1)
                }) + "\\n"

    return StreamingResponse(gen(), media_type="application/x-ndjson")


# =====================================================================
# 3. AVATAR ENDPOINTS (Forwarded directly to MuseTalk on Port 8001)
# =====================================================================
@app.post("/register_avatar")
async def register_avatar(
    avatar_id: str = Form("dadaji"),
    bbox_shift: int = Form(0),
    file: UploadFile = File(...)
):
    \"\"\"Proxies avatar registration to MuseTalk on port 8001.\"\"\"
    content = await file.read()
    filename = file.filename or "media.mp4"

    try:
        async with httpx.AsyncClient(timeout=300.0) as client:
            files = {"file": (filename, content, file.content_type or "application/octet-stream")}
            data = {"avatar_id": avatar_id, "bbox_shift": str(bbox_shift)}
            resp = await client.post(f"{MUSETALK_INTERNAL_URL}/register_avatar", data=data, files=files)
            return JSONResponse(status_code=resp.status_code, content=resp.json())
    except httpx.ConnectError:
        raise HTTPException(status_code=503, detail="MuseTalk backend server (port 8001) is not running.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Proxy error to MuseTalk: {e}")


@app.post("/lipsync_stream")
async def lipsync_stream(
    avatar_id: str = Form("dadaji"),
    audio: UploadFile = File(...)
):
    \"\"\"Proxies real-time streaming lip-sync to MuseTalk on port 8001.\"\"\"
    audio_bytes = await audio.read()
    filename = audio.filename or "audio.wav"

    async def forward_stream():
        client = httpx.AsyncClient(timeout=180.0)
        try:
            files = {"audio": (filename, audio_bytes, "audio/wav")}
            data = {"avatar_id": avatar_id}
            async with client.stream("POST", f"{MUSETALK_INTERNAL_URL}/lipsync_stream", data=data, files=files) as response:
                if response.status_code != 200:
                    yield json.dumps({"error": f"MuseTalk status {response.status_code}"}) + "\\n"
                    return
                async for line in response.aiter_lines():
                    if line:
                        yield line + "\\n"
        finally:
            await client.aclose()

    return StreamingResponse(forward_stream(), media_type="application/x-ndjson")


@app.post("/lipsync_file")
async def lipsync_file(
    avatar_id: str = Form("dadaji"),
    audio: UploadFile = File(...)
):
    \"\"\"Proxies full MP4 video generation to MuseTalk on port 8001.\"\"\"
    audio_bytes = await audio.read()
    filename = audio.filename or "audio.wav"

    try:
        async with httpx.AsyncClient(timeout=300.0) as client:
            files = {"audio": (filename, audio_bytes, "audio/wav")}
            data = {"avatar_id": avatar_id}
            resp = await client.post(f"{MUSETALK_INTERNAL_URL}/lipsync_file", data=data, files=files)
            if resp.status_code != 200:
                return JSONResponse(status_code=resp.status_code, content={"detail": resp.text})
            return Response(
                content=resp.content,
                media_type="video/mp4",
                headers={"Content-Disposition": f'inline; filename="{avatar_id}_talking.mp4"'}
            )
    except httpx.ConnectError:
        raise HTTPException(status_code=503, detail="MuseTalk backend server (port 8001) is not running.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Proxy error to MuseTalk: {e}")


@app.get("/idle_frame")
async def get_idle_frame(avatar_id: str = Query("dadaji"), frame_index: int = Query(0)):
    \"\"\"Proxies idle frame retrieval to MuseTalk on port 8001.\"\"\"
    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            resp = await client.get(
                f"{MUSETALK_INTERNAL_URL}/idle_frame",
                params={"avatar_id": avatar_id, "frame_index": frame_index}
            )
            if resp.status_code == 200:
                return Response(content=resp.content, media_type="image/jpeg")
            return JSONResponse(status_code=resp.status_code, content={"detail": resp.text})
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Proxy error to MuseTalk: {e}")


# =====================================================================
# 4. UNIFIED ALL-IN-ONE PIPELINE: Direct Speech-To-Avatar (0ms Network Audio)
# =====================================================================
class SpeechAvatarRequest(BaseModel):
    text: str
    speaker_name: str = "default"
    avatar_id: str = "dadaji"
    num_step: int = 16
    stream: bool = False


@app.post("/synthesize_and_lipsync")
async def synthesize_and_lipsync(req: SpeechAvatarRequest):
    \"\"\"
    All-In-One Pipeline:
    1. Synthesizes voice in-memory with OmniVoice directly on GPU.
    2. Sends the in-memory audio directly to MuseTalk on localhost.
    3. Streams back video frames or returns talking MP4 with ZERO internet audio latency!
    \"\"\"
    text = req.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty.")

    target = get_target_voice(req.speaker_name)
    arr = synth_audio_tensor(text, target, num_step=req.num_step)
    wav_bytes = tensor_to_wav_bytes(arr, 24000)

    if req.stream:
        # Stream NDJSON frames
        async def forward_stream():
            client = httpx.AsyncClient(timeout=180.0)
            try:
                files = {"audio": ("speech.wav", wav_bytes, "audio/wav")}
                data = {"avatar_id": req.avatar_id}
                async with client.stream("POST", f"{MUSETALK_INTERNAL_URL}/lipsync_stream", data=data, files=files) as response:
                    async for line in response.aiter_lines():
                        if line:
                            yield line + "\\n"
            finally:
                await client.aclose()
        return StreamingResponse(forward_stream(), media_type="application/x-ndjson")
    else:
        # Return MP4 file
        async with httpx.AsyncClient(timeout=300.0) as client:
            files = {"audio": ("speech.wav", wav_bytes, "audio/wav")}
            data = {"avatar_id": req.avatar_id}
            resp = await client.post(f"{MUSETALK_INTERNAL_URL}/lipsync_file", data=data, files=files)
            if resp.status_code != 200:
                raise HTTPException(status_code=resp.status_code, detail=resp.text)
            return Response(
                content=resp.content,
                media_type="video/mp4",
                headers={"Content-Disposition": f'inline; filename="{req.avatar_id}_talking.mp4"'}
            )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
"""
with open(f'{BASE_DIR}/unified_colab_server.py', 'w', encoding='utf-8') as f:
    f.write(unified_server_code)

print("✅ Server scripts deployed successfully.")

# 4. Start MuseTalk Server in background on Port 8001
print("🚀 [1/3] Starting MuseTalk Neural Lip-Sync Engine on Port 8001...")
musetalk_env = os.environ.copy()
musetalk_env['PORT'] = '8001'
musetalk_env['MUSETALK_VERSION'] = 'v15' if musetalk_version == 'v1.5' else 'v1'
musetalk_env['HF_HOME'] = f'{BASE_DIR}/cache/huggingface'

musetalk_proc = subprocess.Popen(
    [f"{BASE_DIR}/env/bin/python", "-u", f"{MUSETALK_DIR}/colab_musetalk_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=musetalk_env
)

# Wait for MuseTalk readiness
print("⏳ Initializing MuseTalk neural weights in GPU memory...")
musetalk_ready = False
for _ in range(60):
    try:
        req = urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2)
        if req.status == 200:
            info = json.loads(req.read().decode())
            print(f"✅ MuseTalk Engine Ready! Device: {info.get('device')} | VRAM: {info.get('vram_gb')} GB")
            musetalk_ready = True
            break
    except Exception:
        time.sleep(1)

if not musetalk_ready:
    print("⚠️ MuseTalk startup waiting timed out, continuing to launch gateway...")

# 5. Start Unified Server on Port 8000 (Hosts OmniVoice + Proxies MuseTalk)
print("\n🚀 [2/3] Starting Unified Gateway & OmniVoice Engine on Port 8000...")
unified_env = os.environ.copy()
unified_env['MUSETALK_INTERNAL_URL'] = 'http://127.0.0.1:8001'
unified_env['HF_HOME'] = f'{BASE_DIR}/cache/huggingface'

unified_proc = subprocess.Popen(
    [sys.executable, "-u", f"{BASE_DIR}/unified_colab_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=unified_env
)

# Wait for Unified server readiness
print("⏳ Initializing OmniVoice zero-shot voice cloner...")
unified_ready = False
for _ in range(60):
    try:
        req = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        if req.status == 200:
            info = json.loads(req.read().decode())
            print(f"✅ Unified Server Ready! Voice: {info.get('voice_ready')} | Avatar: {info.get('avatar_ready')}")
            unified_ready = True
            break
    except Exception:
        time.sleep(1)

# 6. Establish Single Public Tunnel on Port 8000
print("\n🌐 [3/3] Establishing Single Public Tunnel on Port 8000...")
public_url = ""
if "Cloudflare" in tunnel_provider:
    cf_bin = f'{BASE_DIR}/bin/cloudflared'
    if not os.path.exists(cf_bin):
        os.makedirs(f'{BASE_DIR}/bin', exist_ok=True)
        print("📥 Downloading cloudflared binary...")
        !curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o {cf_bin}
        !chmod +x {cf_bin}
    else:
        print(f"⚡ Using cached cloudflared binary at {cf_bin}")

    cl_proc = subprocess.Popen(
        [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{public_port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    import re
    for _ in range(50):
        line = cl_proc.stdout.readline()
        if line:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                public_url = m.group(0).strip()
                break
        time.sleep(0.4)

    if not public_url:
        from pycloudflared import try_cloudflare
        cl_tunnel = try_cloudflare(port=public_port)
        public_url = getattr(cl_tunnel, "tunnel", getattr(cl_tunnel, "tunnel_url", str(cl_tunnel[0]))).strip().rstrip("/")
else:
    from pyngrok import ngrok
    token = ngrok_auth_token.strip()
    if token:
        ngrok.set_auth_token(token)
    else:
        print("⚠️ Warning: No ngrok token provided. Get one from https://dashboard.ngrok.com")
    ng_tunnel = ngrok.connect(public_port, "http")
    public_url = ng_tunnel.public_url.strip().rstrip("/")

# 7. Display Unified Connection Banner
print("\n" + "=" * 76)
print("🎉 KIN-AI UNIFIED VOICE & AVATAR GPU SERVER IS ONLINE!")
print("=" * 76)
print(f"\n🔗 SINGLE PUBLIC TUNNEL URL: \033[1;32m{public_url}\033[0m")
print("\n👉 Paste this link into your Backend/.env:")
print(f"   COLAB_SERVER_URL = '{public_url}'")
print(f"   KAGGLE_SERVER_URL = '{public_url}'")
print("=" * 76)
print("\nℹ️ Server is streaming live logs below (Press Stop button to terminate):\n")

try:
    while True:
        line = unified_proc.stdout.readline()
        if not line and unified_proc.poll() is not None:
            break
        if line:
            print("[Gateway] ", line.rstrip())
except KeyboardInterrupt:
    print("\n🛑 Server stopped by user.")
    musetalk_proc.terminate()
    unified_proc.terminate()


In [ ]:
#@title 💾 (Kaggle Only) Backup Models to a Kaggle Dataset (Permanent 0-Second Setup)
#@markdown If you want to use these models across multiple notebooks or share them, this archives your cached models into `/kaggle/working/kin_avatar_models`.

import os, shutil

IS_KAGGLE = os.path.exists('/kaggle')
if not IS_KAGGLE:
    print("ℹ️ This helper is designed for Kaggle. On Google Colab, Google Drive or local caching is used.")
else:
    MODELS_DIR = '/kaggle/working/MuseTalk/models'
    OUTPUT_DIR = '/kaggle/working/kin_avatar_models'

    if os.path.exists(MODELS_DIR):
        print(f"📦 Packaging model weights from {MODELS_DIR}...")
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        for item in os.listdir(MODELS_DIR):
            s = os.path.join(MODELS_DIR, item)
            d = os.path.join(OUTPUT_DIR, item)
            if os.path.isdir(s) and not os.path.exists(d):
                shutil.copytree(s, d)
            elif os.path.isfile(s) and not os.path.exists(d):
                shutil.copy2(s, d)

        print("=" * 70)
        print("🎉 Models packaged successfully into: /kaggle/working/kin_avatar_models")
        print("=" * 70)
        print("👉 To use as an instant 0-second Kaggle Dataset:")
        print("   1. Click 'Save Version' on top-right -> 'Save & Run All (Commit)'.")
        print("   2. Go to the completed version -> Click 'Output' -> 'New Dataset'.")
        print("   3. Name it 'kin-avatar-models'.")
        print("   4. In any future notebook, click '+ Add Input' -> add 'kin-avatar-models'.")
        print("   The notebook will automatically detect it and start in 0 seconds with NO downloads!")
    else:
        print("⚠️ Models directory not found. Please run Step 1 first to download the weights.")


In [ ]:
#@title 🎬 (Optional) Step 4: Quick In-Notebook Test (Text ➔ Voice ➔ Video)
#@markdown Test the full pipeline directly inside Kaggle / Colab:

test_text = "Hello! I am your AI avatar, running on a single GPU server." #@param {type:"string"}
speaker_name = "default" #@param {type:"string"}
avatar_id = "test_avatar" #@param {type:"string"}

import os, requests
from IPython.display import display, HTML
from base64 import b64encode

SERVER_URL = "http://127.0.0.1:8000"

print("🔍 Checking Unified Server Health...")
r = requests.get(f"{SERVER_URL}/health", timeout=5)
print("Server status:", r.json())

cached_avatars = r.json().get("cached_avatars", [])
if not cached_avatars:
    print("⚠️ No cached avatars found. Please register an avatar via the Kin-AI web interface or upload a portrait.")
else:
    avatar_id = cached_avatars[0]
    print(f"⚡ Using existing cached avatar: '{avatar_id}'")

    print(f"\n🎙️ Synthesizing voice and rendering avatar video for: '{test_text}'...")
    res = requests.post(
        f"{SERVER_URL}/synthesize_and_lipsync",
        json={"text": test_text, "speaker_name": speaker_name, "avatar_id": avatar_id, "num_step": 16, "stream": False},
        timeout=180
    )
    res.raise_for_status()

    IS_KAGGLE = os.path.exists('/kaggle')
    BASE_DIR = '/kaggle/working' if IS_KAGGLE else ('/content' if os.path.exists('/content') else os.path.abspath('.'))
    out_path = f"{BASE_DIR}/unified_output.mp4"
    with open(out_path, "wb") as f:
        f.write(res.content)

    print(f"\n🎉 Generated Video saved ({len(res.content) // 1024} KB)!")
    mp4 = open(out_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
    <video width="480" height="480" controls autoplay style="border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
